In [1]:
!pip install 'pydantic[email]' #para o código rodar no notebook foi necessário fazer essa instalação pois apareciam erros de execução ao tentar validar o email

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 15.6 MB/s eta 0:00:00


Minha ideia é utilizar os conceitos aprendidos para criar um validador de usuário de um sistema acadêmica, nesse caso estou pensando no sigaa como base. Acredito que vai ser ideal pois serão cadastradas e validadas informações desse usuário e ele precisa necessariamente ser de alguma dos tipos definidos podendo ser: professor, aluno, admin e coordenador, espero utilizar todos os conceitos em um único código para tornar a validação dos dados robustas.

In [6]:
import enum #fiz a unificação dos imports que encontramos no exemplo 3 do Arjan já que não haveria problemas e minha intenção é fazer o máximo de testes e validações no mesmo código
import hashlib
import re
from typing import Any, Self

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    ValidationError,
    field_serializer,
    field_validator,
    model_serializer,
    model_validator,
    SecretStr,
)

VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$") #utilizei o mesmo principio do REGEX já que a ideia é aplicar em uma suposta validação de dados no login de usuário de sistema acadêmico
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")


class User_type(enum.IntFlag): #aqui implementei mudanças para se adequar a minha ideia e adicionei os tipos de usuário possíveis em um sistema academico
    Student = 1
    Teacher = 2
    Admin = 4
    Coordinator = 8


class User(BaseModel): #as regras para a classe de usuário se mantem
    name: str = Field(examples=["Karol"])
    email: EmailStr = Field(
        examples=["user@gmail.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"],
        description="The password of the user",
        exclude=True,
    )
    user_type: User_type = Field(
        description="The type of the user",
        examples=[1, 2, 4, 8],
        default=User_type.Student,
        validate_default=True, #apesar de discordar da ideia do email se fixo, aqui eu mantive pois já que a ideia é o login dos sistema acadêmico, o email institucional é único
    )


    @field_validator("name") #é aplicada a regra de validação do nome
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

    @field_validator("user_type", mode="before") #é aplicada a regra de validação do tipo de usuário
    @classmethod
    def validate_user_type(cls, v: int | str | User_type) -> User_type:
        op = {
            int: lambda x: User_type(x),
            str: lambda x: User_type[x],
            User_type: lambda x: x,
        }
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"User type is invalid, please use one of the following: {', '.join([x.name for x in User_type])}"
            )

    @model_validator(mode="before") #apesar de em tese não usarmos essa primeira parte para nossos testes, fiz um esqueleto único e aqui se aplicam as validações de usuário pré e pós envio de dados
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")

        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")

        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )

        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

    @model_validator(mode="after")
    def validate_user_post(self) -> Self:
        if self.user_type == User_type.Admin and self.name != "Karol":
            raise ValueError("Only Karol can be an admin")
        return self


    @field_serializer("user_type", when_used="json") #aqui entram as regras de serialização que foram apresentadas
    def serialize_user_type(self, v) -> str:
        return v.name

    @model_serializer(mode="wrap", when_used="json")
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {"name": self.name, "user_type": self.user_type.name}
        return serializer(self)


def validate(data: dict[str, Any]) -> None: #é criada a função para validar os dados que será utilizada nesses primeiros testes que foram apresentados
    try:
        user = User.model_validate(data)
        print("Valid user:")
        print(user.model_dump())
    except ValidationError as e:
        print("User is invalid:")
        print(e)


def main() -> None: #foram criados dados que servirão como base para os testes que serão feitos
    test_data = {
        "good_data": {
            "name": "Karol",
            "email": "karol@gmail.com",
            "password": "Password123",
            "user_type": "Admin",
        },
        "bad_user_type": {
            "name": "Karol",
            "email": "karol@gmail.com",
            "password": "Password123",
            "user_type": "usuario",
        },
        "bad_data": {
            "name": "Karol",
            "email": "bad email",
            "password": "bad",
        },
        "bad_name": {
            "name": "Karol<-_->",
            "email": "karol@gmail.com",
            "password": "Password123",
        },
        "duplicate": {
            "name": "Karol",
            "email": "karol@gmail.com",
            "password": "Karol123",
        },
        "missing_data": {
            "email": "bad",
            "password": "bad",
        },
    }

    for example_name, data in test_data.items(): #apresenta os resultados na tela após rodar o código
        print(f"\n {example_name}")
        validate(data)


if __name__ == "__main__":
    main()


 good_data
Valid user:
{'name': 'Karol', 'email': 'karol@gmail.com', 'user_type': <User_type.Admin: 4>}

 bad_user_type
User is invalid:
1 validation error for User
user_type
  Value error, User type is invalid, please use one of the following: Student, Teacher, Admin, Coordinator [type=value_error, input_value='usuario', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

 bad_data
User is invalid:
1 validation error for User
  Value error, Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number [type=value_error, input_value={'name': 'Karol', 'email'...ail', 'password': 'bad'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

 bad_name
User is invalid:
1 validation error for User
name
  Value error, Name is invalid, must contain only letters and be at least 2 characters long [type=value_error, input_value='Karol<-_->', input_type=str]
    For further information

Não encontrei uma forma de fazer os testes que são apresentados no vídeo de forma unificada, logo precisei separar os códigos, mas a base acabou sendo praticamente a mesma, mudei a função principal de um código para o outro

In [5]:
import enum
import hashlib
import re
from typing import Any, Self

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    ValidationError,
    field_serializer,
    field_validator,
    model_serializer,
    model_validator,
    SecretStr,
)

VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")


class User_type(enum.IntFlag):
    Student = 1
    Teacher = 2
    Admin = 4
    Coordinator = 8


class User(BaseModel):
    name: str = Field(examples=["Karol"])
    email: EmailStr = Field(
        examples=["user@gmail.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"],
        description="The password of the user",
        exclude=True,
    )
    user_type: User_type = Field(
        description="The type of the user",
        examples=[1, 2, 4, 8],
        default=User_type.Student,
        validate_default=True,
    )

    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

    @field_validator("user_type", mode="before")
    @classmethod
    def validate_user_type(cls, v: int | str | User_type) -> User_type:
        op = {
            int: lambda x: User_type(x),
            str: lambda x: User_type[x],
            User_type: lambda x: x,
        }
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"User type is invalid, please use one of the following: {', '.join([x.name for x in User_type])}"
            )

    @model_validator(mode="before")
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")

        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")

        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )

        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

    @model_validator(mode="after")
    def validate_user_post(self) -> Self:
        if self.user_type == User_type.Admin and self.name != "Karol":
            raise ValueError("Only Karol can be an admin")
        return self


    @field_serializer("user_type", when_used="json")
    def serialize_user_type(self, v) -> str:
        return v.name

    @model_serializer(mode="wrap", when_used="json")
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {"name": self.name, "user_type": self.user_type.name}
        return serializer(self)


def main() -> None: #aqui entram as mudanças para o segundo tipo de teste, crio o dado e imprimo na tela as serializações criadas
    data = {
        "name": "Karol",
        "email": "karol@gmail.com",
        "password": "Password123",
        "user_type": "Admin",
    }

    try:
        user = User.model_validate(data)

        print(
            "The serializer that returns a dict:",
            user.model_dump(),
            sep="\n",
            end="\n\n",
        )

        print(
            "The serializer that returns a JSON string:",
            user.model_dump(mode="json"),
            sep="\n",
            end="\n\n",
        )

        print(
            "The serializer that returns a json string, excluding the user_type:",
            user.model_dump(exclude=["user_type"], mode="json"),
            sep="\n",
            end="\n\n",
        )

        print(
            "The serializer that encodes all values to a dict:",
            dict(user),
            sep="\n",
        )

    except ValidationError as e:
        print("User is invalid:")
        print(e)


if __name__ == "__main__":
    main()

The serializer that returns a dict:
{'name': 'Karol', 'email': 'karol@gmail.com', 'user_type': <User_type.Admin: 4>}

The serializer that returns a JSON string:
{'name': 'Karol', 'user_type': 'Admin'}

The serializer that returns a json string, excluding the user_type:
{'name': 'Karol', 'email': 'karol@gmail.com'}

The serializer that encodes all values to a dict:
{'name': 'Karol', 'email': 'karol@gmail.com', 'password': SecretStr('**********'), 'user_type': <User_type.Admin: 4>}
